# VascuMap Batch Pipeline

Runs the full VascuMap pipeline on every image in a source folder.  
- **LIF files**: iterates through each image within the file  
- **TIF/TIFF files**: one pipeline run per file  
- Per-image output folder under `Z:\Bel\Vascumap_Outputs\<image_name>\`

**Default outputs** (always saved):
| File | Description |
|------|-------------|
| `*_overlay_geometry_0.tif` | 2-D device overlay with inner/outer geometry |
| `*_skeleton_overview.png` | Skeleton overview plot |
| `*_analysis_metrics.csv` | Quantitative vascular network metrics |

**Extra outputs** (when `save_all_interim = True`):
| File | Description |
|------|-------------|
| `*_cropped_stack_aligned.npy` | Brightfield stack at 2 µm iso |
| `*_vessel_translation_aligned.npy` | Pix2Pix image translation |
| `*_clean_segmentation.npy` | Cleaned binary vessel segmentation |
| `*_skeleton.npy` | Skeleton from pruned graph |
| `*_holes.npy` | Binary pore mask |
| `*_hole_labels_per_slice.npy` | Per-slice labelled pores |
| `*_hole_distance_per_slice_um.npy` | Inscribed-radius distance map |
| `*_full_graph_skeleton.npy` | Pre-pruning skeleton |
| `*_vessel_mask.npy` | Raw vessel mask |
| `*_graph_nodes.npz` | Sprout/junction node coords |
| `*_clean_graph.pkl` | NetworkX graph (pickle) |

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add bel_vascumap to the path so we can import the pipeline
sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))

import numpy as np
import tifffile
from liffile import LifFile

from vascumap import VascuMap
from utils import resize_dask

W0319 20:53:30.280000 87344 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ── Load models once (shared across all batch jobs) ────────────────────────────
from models import Pix2Pix, load_segmentation_model

shared_model_p2p = Pix2Pix(
    model_path=r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\epoch=117-val_g_psnr=20.47-val_g_ssim=0.62.ckpt"
)
shared_model_unet = load_segmentation_model(
    r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\best_full.pth"
)
print("Models loaded.")

Models loaded.


In [ ]:
# ── Job discovery helper ──────────────────────────────────────────────────────
def discover_jobs(source_dir: str, force_mask_central_region=None, require_merged: bool = True):
    """Discover .lif/.tif/.tiff files and return batch jobs.

    Returns
    -------
    source : Path
        Resolved source directory.
    image_files : list[Path]
        Matching image files found in source.
    jobs : list[tuple[Path, int, bool]]
        (path, image_index, should_mask) entries for batch execution.
    """
    source = Path(source_dir)
    if not source.is_dir():
        raise FileNotFoundError(f"source_dir does not exist: {source}")

    image_files = sorted(
        p for p in source.iterdir()
        if p.is_file() and p.suffix.lower() in (".lif", ".tif", ".tiff")
    )

    jobs = []
    for p in image_files:
        should_mask = (
            force_mask_central_region
            if force_mask_central_region is not None
            else ("marina" in p.name.lower())
        )

        if p.suffix.lower() == ".lif":
            try:
                with LifFile(p) as lif:
                    n_images = len(lif.images)
                    for idx in range(n_images):
                        img_name = lif.images[idx].name if hasattr(lif.images[idx], "name") else ""
                        if require_merged and "merged" not in img_name.lower():
                            continue
                        jobs.append((p, idx, should_mask))
            except Exception as exc:
                print(f"[SKIP] Could not inspect {p.name}: {exc}")
        else:
            if require_merged and "merged" not in p.name.lower():
                continue
            jobs.append((p, 0, should_mask))

    return source, image_files, jobs

In [ ]:
# ── Processing + batch execution helpers ──────────────────────────────────────
def process_and_save(image_path: Path, image_index: int, output_base: str,
                     device_width_um: float,
                     hough_line_length: int = 400, ####250 works better for maria
                     mask_central_region: bool = True,
                     save_all_interim: bool = False,
                     channel: int = 0,
                     model_p2p=None,
                     model_unet=None):
    """Run full VascuMap pipeline and save aligned outputs to a per-image folder."""
    vm = VascuMap(
        use_device_segmentation_app=False,
        image_source_path=str(image_path),
        image_index=image_index,
        device_width_um=device_width_um,
        hough_line_length=hough_line_length,
        mask_central_region=mask_central_region,
        channel=channel,
        model_p2p=model_p2p,
        model_unet=model_unet,
    )

    if image_path.suffix.lower() == ".lif":
        vm.image_name = f"{image_path.stem}_img{image_index}_{vm.image_name or 'image'}"
    else:
        vm.image_name = f"{image_path.stem}_{vm.image_name or 'image'}"

    output_dir = Path(output_base) / vm.image_name
    print(f"  Output → {output_dir}")
    vm.pipeline(output_dir=output_dir, save_all_interim=save_all_interim)
    return vm


def run_batch(jobs, output_base: str, device_width_um: float, hough_line_length: int = 400, channel: int = 0, save_all_interim: bool = False,
              model_p2p=None, model_unet=None, max_retries: int = 2, label: str = "Batch"):
    """Run a list of jobs and print a summary."""
    results = []  # (short_name, status, message)

    for i, (image_path, image_index, mask_flag) in enumerate(jobs, start=1):
        tag = f" (LIF #{image_index})" if image_path.suffix.lower() == ".lif" else ""
        print(f"\n{'='*70}")
        print(f"[{i}/{len(jobs)}] {image_path.name}{tag}  mask_central_region={mask_flag}")
        print(f"{'='*70}")

        last_exc = None
        for attempt in range(1, max_retries + 1):
            try:
                vm = process_and_save(
                    image_path=image_path,
                    image_index=image_index,
                    output_base=output_base,
                    device_width_um=device_width_um,
                    hough_line_length=hough_line_length,
                    mask_central_region=mask_flag,
                    save_all_interim=save_all_interim,
                    channel=channel,
                    model_p2p=model_p2p,
                    model_unet=model_unet,
                )
                results.append((vm.image_name or image_path.stem, "OK", ""))
                last_exc = None
                break
            except Exception as exc:
                last_exc = exc
                if attempt < max_retries:
                    print(f"  ⚠ Attempt {attempt} failed: {exc}  — retrying...")
                else:
                    print(f"  ✗ FAILED after {max_retries} attempts: {exc}")

        if last_exc is not None:
            results.append((image_path.name + tag, "FAILED", str(last_exc)))

    print(f"\n{'='*70}")
    n_ok = sum(1 for _, s, _ in results if s == "OK")
    print(f"{label} complete: {n_ok}/{len(results)} succeeded")
    if any(s == "FAILED" for _, s, _ in results):
        print("\nFailed images:")
        for name, status, msg in results:
            if status == "FAILED":
                print(f"  - {name}: {msg}")

    return results

In [ ]:
# ── Run configuration (edit here) ─────────────────────────────────────────────
## Active batch target (edit these)
source_dir = r"Z:\Bel\Vascumap_Example_Lifs\original_images\test_subset"
output_base = r"Z:\Bel\Vascumap_Example_Lifs\outputs"
force_mask_central_region = None   # True / False / None (auto by filename)
require_merged = False              # True only if names include 'merged'
save_all_interim = False

## Shared processing settings
device_width_um = 35.0
hough_line_length = 30             # Default (previous behavior); lower to override
channel = 0
max_retries = 2

In [ ]:
# ── Discover jobs + run ────────────────────────────────────────────────────────
source, image_files, jobs = discover_jobs(
    source_dir=source_dir,
    force_mask_central_region=force_mask_central_region,
    require_merged=require_merged,
    )

print(f"Source: {source}")
print(f"Files found: {len(image_files)}  |  Total jobs: {len(jobs)}")
if len(image_files) == 0:
    print("No .lif/.tif/.tiff files found in this folder.")
if len(jobs) == 0 and len(image_files) > 0:
    print("Files were found, but all were filtered out (likely by require_merged).")

for i, (p, idx, _) in enumerate(jobs, start=1):
    tag = f" (LIF image {idx})" if p.suffix.lower() == ".lif" else ""
    print(f"  {i}. {p.name}{tag}")

maria_results = run_batch(
    jobs=jobs[9:],
    output_base=output_base,
    device_width_um=device_width_um,
    hough_line_length=hough_line_length,
    channel=channel,
    save_all_interim=save_all_interim,
    model_p2p=shared_model_p2p,
    model_unet=shared_model_unet,
    max_retries=max_retries,
    label="Batch",
)

Source: Z:\Bel\Vascumap_Example_Lifs\original_images\test_subset
Files found: 9  |  Total jobs: 43
  1. 20250619_Vascumap_Dev25_11_Daisy10.tif
  2. 20250619_Vascumap_Dev25_11_WicellWicellECFB10.tif
  3. Akinola_09102024_day7_hmvec_cs_merged.lif (LIF image 0)
  4. Akinola_09102024_day7_hmvec_cs_merged.lif (LIF image 1)
  5. Akinola_09102024_day7_hmvec_cs_merged.lif (LIF image 2)
  6. Akinola_09102024_day7_hmvec_cs_merged.lif (LIF image 3)
  7. Akinola_09102024_day7_hmvec_cs_merged.lif (LIF image 4)
  8. Akinola_09102024_day7_hmvec_cs_merged.lif (LIF image 5)
  9. Defne_placenta_270325_Dexa2_Device_2.tif
  10. Defne_placenta_270325_Gly_Device_6.tif
  11. Dorota_merged_stellaris_R1_007.tif
  12. Dorota_merged_stellaris_R1_017.tif
  13. farid_new_flow_double_channel.tif
  14. marina_M4_2024.10.24_FL2_FL6.lif (LIF image 0)
  15. marina_M4_2024.10.24_FL2_FL6.lif (LIF image 1)
  16. marina_M4_2024.10.24_FL2_FL6.lif (LIF image 2)
  17. marina_M4_2024.10.24_FL2_FL6.lif (LIF image 3)
  18. marin

Processing chunks: 100%|██████████| 3/3 [00:44<00:00, 14.95s/it]


strong contiguous vote planes 10-19
Running skeletonisation and analysis...


Processing chunks: 100%|██████████| 1/1 [00:08<00:00,  8.19s/it]


 chip_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_orientation_deg  p90_minus_p10_sprout_and_branch_orientation_deg  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  average_vessel_volume_um3  total_internal_pore_count  internal_pore_area_fraction_in_filled_vascular_area  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  median_internal_pore_max_inscribed_radius_um  p90_minus_p10_internal_pore_max_inscribed_radius_um  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3  total_internal_pore_density_

...

Processing chunks: 100%|██████████| 3/3 [00:26<00:00,  8.92s/it]


strong contiguous vote planes 0-9
Running skeletonisation and analysis...


Processing chunks: 100%|██████████| 1/1 [00:04<00:00,  4.56s/it]


 chip_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_orientation_deg  p90_minus_p10_sprout_and_branch_orientation_deg  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  average_vessel_volume_um3  total_internal_pore_count  internal_pore_area_fraction_in_filled_vascular_area  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  median_internal_pore_max_inscribed_radius_um  p90_minus_p10_internal_pore_max_inscribed_radius_um  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3  total_internal_pore_density_

 61%|██████    | 439/718 [04:10<02:39,  1.75it/s]